In [1]:
import requests
import urllib3
import numpy as np
from tensorflow import keras

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
print("Librerie importate!")

Librerie importate!


In [2]:
def scarica_testo_dante():
    url = "https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt"
    print(f"Scaricamento da: {url}...")
    try:
        response = requests.get(url, verify=False)
        response.raise_for_status()
        response.encoding = 'utf-8'
        text = response.text
        print(f"Download completato! Lunghezza: {len(text)} caratteri")
        return text
    except Exception as e:
        print(f"Errore: {e}, provo link backup...")
        url_backup = "https://raw.githubusercontent.com/wpm/t-sne-text-vis/master/data/divina_commedia.txt"
        r2 = requests.get(url_backup, verify=False)
        r2.encoding = 'utf-8'
        return r2.text

testo = scarica_testo_dante()

Scaricamento da: https://dmf.unicatt.it/~della/pythoncourse18/commedia.txt...
Download completato! Lunghezza: 551846 caratteri


In [3]:
caratteri_unici = sorted(set(testo))
num_car = len(caratteri_unici)
char_to_idx = {c: i for i, c in enumerate(caratteri_unici)}
idx_to_char = {i: c for i, c in enumerate(caratteri_unici)}
print(f"Caratteri unici: {num_car}")

Caratteri unici: 69


In [4]:
seq_length = 80
X, y = [], []
for i in range(0, len(testo) - seq_length):
    sequenza = testo[i:i + seq_length]
    carattere_successivo = testo[i + seq_length]
    X.append([char_to_idx[c] for c in sequenza])
    y.append(char_to_idx[carattere_successivo])

X = np.reshape(X, (len(X), seq_length, 1))
X = X / num_car
y = keras.utils.to_categorical(y, num_classes=num_car)
print(f"Dataset creato! Shape X: {X.shape}")

Dataset creato! Shape X: (551766, 80, 1)


In [5]:
modello = keras.Sequential([
    keras.layers.LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True),
    keras.layers.Dropout(0.2),
    keras.layers.LSTM(256),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(num_car, activation='softmax')
])
modello.compile(loss='categorical_crossentropy', optimizer='adam')
modello.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 80, 256)        │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 80, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 69)             │        17,733 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 807,237 (3.08 MB)

 Trainable params: 807,237 (3.08 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
print("Inizio addestramento...")
storico = modello.fit(X, y, epochs=10, batch_size=128, verbose=1)
print("Addestramento completato!")

Inizio addestramento...
Epoch 1/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 116s 26ms/step - loss: 2.6182
Epoch 2/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 121s 28ms/step - loss: 2.3513
Epoch 3/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 121s 28ms/step - loss: 2.2143
Epoch 4/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 121s 28ms/step - loss: 2.1089
Epoch 5/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 121s 28ms/step - loss: 2.0246
Epoch 6/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.9647
Epoch 7/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.9185
Epoch 8/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.8827
Epoch 9/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.8529
Epoch 10/10
4311/4311 ━━━━━━━━━━━━━━━━━━━━ 120s 28ms/step - loss: 1.8272
Addestramento completato!


In [14]:
# Generazione testo con temperatura
def genera_testo(modello, testo_seed, char_to_idx, idx_to_char,
                 num_car, temperatura=1.0, lunghezza=200):

    # Padding automatico: se il seed è corto lo allunghiamo
    while len(testo_seed) < seq_length:
        testo_seed = " " + testo_seed

    # Prendiamo esattamente gli ultimi 80 caratteri
    testo_seed = testo_seed[-seq_length:]

    testo_generato = testo_seed
    sequenza = testo_seed

    for _ in range(lunghezza):
        x = [char_to_idx.get(c, 0) for c in sequenza[-seq_length:]]
        x = np.reshape(x, (1, seq_length, 1))
        x = x / num_car

        previsioni = modello.predict(x, verbose=0)[0]
        previsioni = np.log(previsioni + 1e-10) / temperatura
        previsioni = np.exp(previsioni)
        previsioni = previsioni / np.sum(previsioni)

        idx = np.random.choice(len(previsioni), p=previsioni)
        carattere = idx_to_char[idx]

        testo_generato += carattere
        sequenza += carattere

    return testo_generato

# Testiamo con tre temperature diverse
seed = "Nel mezzo del cammin di nostra vita"

print("=" * 50)
print("TEMPERATURA 0.2 (Freddo/Conservativo):")
print("=" * 50)
print(genera_testo(modello, seed, char_to_idx, idx_to_char, num_car, temperatura=0.2))

print("\n" + "=" * 50)
print("TEMPERATURA 1.0 (Standard):")
print("=" * 50)
print(genera_testo(modello, seed, char_to_idx, idx_to_char, num_car, temperatura=1.0))

print("\n" + "=" * 50)
print("TEMPERATURA 2.0 (Caldo/Folle):")
print("=" * 50)
print(genera_testo(modello, seed, char_to_idx, idx_to_char, num_car, temperatura=2.0))

TEMPERATURA 0.2 (Freddo/Conservativo):
                                             Nel mezzo del cammin di nostra vita,
che 'l suo piene si conventa li si fara,
sì che su che 'l suo parto si conpenta.
  Questo si pissa d con la sua visa,
che la colce che la sisa si dontenna,
che la morta che di sua visa sia sia.

TEMPERATURA 1.0 (Standard):
                                             Nel mezzo del cammin di nostra vita.
mer ehrpo fiuso più a' movti, c cheooo
ca m'altro veggelo in te suo gar?
  Enssi m'antoosrr ee la votirne
ch carri cnm altr d'alaro devtero,
quand'io voee tri, che 'l ferm c sé cisre";
  'Qacu

TEMPERATURA 2.0 (Caldo/Folle):
                                             Nel mezzo del cammin di nostra vita?Irf;
raro um mecòeobn ah rgcpntadu'ee' 
ponsism"pì,m'aabhsi 'tgu rrifbuze
aèabìa, g udzne'mrimo gmuiseb!emrua!.
  Saptoc,:sac, 'là neaeodo,i'cdg,
làò DQ'bsenza; d ennrsro inaonis'
oaruiglam:san'


Dopo aver testato tre temperature diverse, la temperatura 1.0 ha prodotto il testo più credibile. Con temperatura 0.2 il modello era troppo ripetitivo, scegliendo sempre le stesse parole. Con temperatura 2.0 il testo era caotico e incomprensibile. La temperatura 1.0 rappresenta il miglior compromesso tra grammatica e creatività poetica.